# Constrained Robotics Control
# Assignment 2

## Initial imports

In [1]:
!pip install uaibot
!pip install numpy
!pip install ipywidgets  # Required for interactive matplotlib plots
!pip install ipympl


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import uaibot as ub
import numpy as np
import matplotlib.pyplot as plt
import random

In [3]:
# Used only to save the problems to be sent via email
save = False

In [4]:
# Create robots
kuka = ub.Robot.create_kuka_lbr_iiwa()
tx60 = ub.Robot.create_staubli_tx60()

# select a robot
robot = kuka

In [5]:
# %matplotlib widget

In [6]:
total_time = 10
dt = 0.01 # Control frequency
steps = int(total_time/dt)

In [7]:
# Useful functions for the assignment
def task_fun(_robot, _q, _htm_tg_t):

    n = np.shape(_q)[0]

    jac, htm_e = _robot.jac_geo(_q)

    x = htm_e[0:3,0]
    y = htm_e[0:3,1]
    z = htm_e[0:3,2]

    s = htm_e[0:3,3]

    xd = _htm_tg_t[0:3,0]
    yd = _htm_tg_t[0:3,1]
    zd = _htm_tg_t[0:3,2]

    sd = np.array(_htm_tg_t[0:3,3]).reshape(3,1)

    jac_v = jac[0:3,:]
    jac_w = jac[3:6,:]

    r = np.matrix(np.zeros((6,1)))

    r[0:3,0] = s-sd
    r[3,0] = 1-xd.T*x
    r[4,0] = 1-yd.T*y
    r[5,0] = 1-zd.T*z

    jac_r = np.matrix(np.zeros((6,n)))

    jac_r[0:3,:] = jac_v
    jac_r[3,:] = xd.T*ub.Utils.S(x)*jac_w
    jac_r[4,:] = yd.T*ub.Utils.S(y)*jac_w
    jac_r[5,:] = zd.T*ub.Utils.S(z)*jac_w

    return r, jac_r


def fun_phi(_r):
    K = 2.0
    _r_mod = np.matrix(np.zeros((6,1)))

    for i in range(6):
        _r_mod[i,0] = -K*np.sqrt(_r[i,0]) if _r[i,0]>=0 else K*np.sqrt(-_r[i,0])

    return _r_mod

def compute_control(_q, _t, _robot, _vel_vec, _dt=0.01):
    """
    _vel_vec must be a 3x1 velocity vector
    """
    jac, htm_curr = _robot.jac_geo(_q)
    x_vec = htm_curr[0:3,0]
    y_vec = htm_curr[0:3,1]
    z_vec = htm_curr[0:3,2]
    s = htm_curr[0:3,3]

    ds = np.array((_vel_vec * _dt)).reshape(3,1) # displacement in position

    htm_tg_t = np.eye(4)
    htm_tg_t[0:3,3] = np.array(htm_curr[0:3,3] + ds).reshape(-1)

    xd = htm_tg_t[0:3, 0]
    yd = htm_tg_t[0:3, 1]
    zd = htm_tg_t[0:3, 2]

    r, jac_r = task_fun(_robot, _q, htm_tg_t)

    dr_dt = np.vstack((
        -_vel_vec, # negative
        -xd.T * x_vec,
        -yd.T * y_vec,
        -zd.T * z_vec,
    ))

    u = ub.Utils.dp_inv(jac_r, 0.01)*(fun_phi(r)-dr_dt)

    return u


def measure_config(_robot):
    return _robot.q


def send_joint_velocity(_robot, _u, _t):
    dt = 0.01
    qprox = _robot.q + _u*dt
    _robot.add_ani_frame(time=_t+dt, q = qprox)


## PROBLEM 1

In [8]:
import numpy as np
import uaibot as ub
import matplotlib.pyplot as plt

target_curve = []
sim = None

#Create the curve as a list of (n,1)-shaped np.matrix
#In this case, n=2
height = 0.5
center = [0.75, 0, height]
radius = 0.15
n_points = 1000

target_ori = ub.Utils.rot([0,0,1], 0) # desired rotation
for i in range(n_points+1):
    t = 2*np.pi*i/(n_points)
    x_curve = center[0] + radius * np.cos(t)
    y_curve = center[1] + radius * np.sin(t)
    z_curve = height
    target_curve.append([x_curve, y_curve, z_curve])

target_curve = np.array(target_curve).reshape(-1, 3)

initial_htm = ub.Utils.trn(target_curve[0])*target_ori
# frame = ub.Frame(htm = initial_htm, size = 0.1) # fisrt target in first position

#Set the starting point
q = measure_config(robot)
jac, htm_e = robot.jac_geo(q)

#Set the integration step
dt = 0.01
#Set the simulation time
t_sim = 15

t = 0

pc = ub.simobjects.PointCloud(
    name="path",
    points=target_curve.T,   # expects 3 x N
    size=0.02,
    color="yellow"
)

sim = ub.Simulation(
    [robot, pc],
    width=600,
    height=600
)

#Simulate the system dot{q} = \Psi_{\mathcal{P}}(q)
hist_q = []
# total_sim_points = round(t_sim/dt)
for i in range(n_points):
    # curve_idx = int((i/total_sim_points)*n_points)
    curve_idx = i
    q = measure_config(robot)
    jac, htm_e = robot.jac_geo(q)
    eff_pos = htm_e[0:3, 3]
    frame_position = target_curve[curve_idx]
    # print(f"curveidx = {curve_idx}")
    # print(f"Frame position = {frame_position}")

    #Return the vector psi, the Euclidean distance to the curve, and
    #the index with closest point to q
    psi, dist, _ = ub.Robot.vector_field(eff_pos,target_curve,alpha=1, is_closed=True)
    #print(f"Current position = {curr_position.T}")
    #print(f"curve_idx = {curve_idx}")
    #print(f"target_curve[curve_idx] = {target_curve[curve_idx]}")
    #print(f"psi = {psi}")
    #print(f"dist = {dist}")

    u = compute_control(q, t, robot, psi)

    send_joint_velocity(robot, u, t)
    hist_q.append(q)

    # frame_htm = ub.Utils.trn(frame_position)
    # frame.add_ani_frame(t, frame_htm)
    pc.add_ani_frame(time=t, initial_ind=0, final_ind=n_points) # plot full circle from the beginning

    t += dt


/tmp/ipykernel_2933735/1856701054.py:26: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  r[3,0] = 1-xd.T*x
/tmp/ipykernel_2933735/1856701054.py:27: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  r[4,0] = 1-yd.T*y
/tmp/ipykernel_2933735/1856701054.py:28: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  r[5,0] = 1-zd.T*z


In [9]:
if save:
    sim.save(".", "problem1")
sim.run()

As expected, this approach prevents the robot from following targets which go throught the robot's own body, for example.

## PROBLEM 2

### 2.1) Implement a second order kinematic control to control the joint acceleration of a manipulator to achieve a constant target pose for the end-effector.

The code should sample a random target pose, a random start configuration and assume zero initial joint velocity .

You dont need to incorporate any constraint like obstacle avoidance and joint limit.

## Quadratic Programming Formulation for Second-Order Kinematic Control

The optimization problem from the image is:

$$\min_u \left\| \frac{\partial r}{\partial q}(q)u + 2K\frac{\partial r}{\partial q}(q)\dot{q} + K^2r(q) + \Gamma(q, \dot{q}) \right\|^2 + \varepsilon\|u\|^2$$

To convert this to standard QP form: $\min_u \frac{1}{2}u^T H u + f^T u$ subject to $A u \geq b$

Let:
- $G = \frac{\partial r}{\partial q}(q)$ (Jacobian matrix, $m \times n$)
- $c = 2K\frac{\partial r}{\partial q}(q)\dot{q} + K^2r(q) + \Gamma(q, \dot{q})$ (constant vector, $m \times 1$)

Expanding the objective:
$$\|Gu + c\|^2 + \varepsilon\|u\|^2 = (Gu + c)^T(Gu + c) + \varepsilon u^T u$$
$$= u^T G^T G u + 2c^T G u + c^T c + \varepsilon u^T u$$
$$= u^T(G^T G + \varepsilon I) u + 2c^T G u + c^T c$$

For QP form:
- **H** = $2(G^T G + \varepsilon I) = 2G^T G + 2\varepsilon I$
- **f** = $2 G^T c = 2 G^T (2K G \dot{q} + K^2r + \Gamma)$
- **A** = (constraints, if any)
- **b** = (constraint bounds, if any)

Note: The constant term $c^T c$ doesn't affect the optimization and can be ignored.


In [10]:
# Recreate robot
robot = ub.Robot.create_kuka_lbr_iiwa()

In [11]:
# Second order kinematic control
# Useful functions for the assignment
def task_fun(_robot, _q, _htm_tg_t):

    #
    n = np.shape(_q)[0]

    jac, htm_e = _robot.jac_geo(_q)

    x = htm_e[0:3,0]
    y = htm_e[0:3,1]
    z = htm_e[0:3,2]

    s = htm_e[0:3,3]

    xd = _htm_tg_t[0:3,0]
    yd = _htm_tg_t[0:3,1]
    zd = _htm_tg_t[0:3,2]

    sd = np.array(_htm_tg_t[0:3,3]).reshape(3,1)

    jac_v = jac[0:3,:]
    jac_w = jac[3:6,:]

    r = np.matrix(np.zeros((6,1)))

    r[0:3,0] = s-sd
    r[3,0] = 1-xd.T*x
    r[4,0] = 1-yd.T*y
    r[5,0] = 1-zd.T*z

    jac_r = np.matrix(np.zeros((6,n)))

    jac_r[0:3,:] = jac_v
    jac_r[3,:] = xd.T*ub.Utils.S(x)*jac_w
    jac_r[4,:] = yd.T*ub.Utils.S(y)*jac_w
    jac_r[5,:] = zd.T*ub.Utils.S(z)*jac_w

    return r, jac_r


def fun_phi(_r):
    K = 2.0
    _r_mod = np.matrix(np.zeros((6,1)))

    for i in range(6):
        _r_mod[i,0] = -K*np.sqrt(_r[i,0]) if _r[i,0]>=0 else K*np.sqrt(-_r[i,0])

    return _r_mod

def compute_control(_q, _qdot, _robot, htm_tg, K=1, eps=1e-2):
    """
    _vel_vec must be a 3x1 velocity vector
    """
    jac, htm_curr = _robot.jac_geo(_q)
    x_vec = htm_curr[0:3,0]
    y_vec = htm_curr[0:3,1]
    z_vec = htm_curr[0:3,2]
    s = htm_curr[0:3,3]

    xd = htm_tg[0:3, 0]
    yd = htm_tg[0:3, 1]
    zd = htm_tg[0:3, 2]

    r, jac_r = task_fun(_robot, _q, htm_tg)

    #print(f"dr_dt = \n{dr_dt}")
    # min || dr * u + 2K * dr(q)*qdot + K²r(q) + T(q,qdot) ||² + e||u||²
    #  u  || dq             dq                             ||
    
    # This is the task function, but I'm creating a 'f' function to make calling
    # it easier in the line that follows it
    f = lambda f_q, f_htm_tg=htm_tg, f_robot=_robot: task_fun(f_robot, f_q, f_htm_tg)[1] # return only 'jac_r'
    
    # Jacobian of the task function (jac_r)
    G = jac_r
    n = robot.q.shape[0] # number of dof
    m = r.shape[0] # dimension of task function 

    Gamma = np.matrix(np.zeros((m,1)))
    # for i in range(Gamma.shape[0]):
    # print(f"i = {i}")
    # print(f"_q = {_q}")
    # print(f"_qdot = {_qdot}")
    # print(f"f(_q + dt*_qdot) shape = {f(_q + dt*_qdot).shape}")
    # print(f"f(_q - dt*_qdot) shape = {f(_q - dt*_qdot).shape}")
    # print(f"f(_q + dt*_qdot) = {f(_q + dt*_qdot)}")
    # print(f"f(_q - dt*_qdot) = {f(_q - dt*_qdot)}")
    # print(f"Gamma[i,:] = {Gamma[i,:]}")
    # print(f"Gamma[i,:] = {(1/2*dt) * (f(_q + dt*_qdot) - f(_q - dt*_qdot)) * _qdot}")
    #                   (m x n)          - (m x n) )         * (n X 1) = (m x 1)
    Gamma = (1/2*dt) * (f(_q + dt*_qdot) - f(_q - dt*_qdot)) * _qdot

    # Compute constant vector c
    c = 2*K*G*_qdot + K**2*r + Gamma
    
    # Compute QP matrices
    # H = 2(G^T G + ε I)
    n_aux = G.shape[1]
    H = 2*(G.T*G + eps*np.eye(n_aux))
    
    # f = 2 G^T c
    f = 2*G.T*c
    
    # A and b are empty if there are no constraints
    # A = np.matrix(np.zeros((0, n)))  # No constraints
    # b = np.matrix(np.zeros((0, 1)))  # No constraints
    
    # print(f"H.shape = {H.shape}")
    # print(f"f.shape = {f.shape}")
    # print(f"A.shape = {A.shape}")
    # print(f"b.shape = {b.shape}")
    A = None
    b = None
    u = ub.Utils.solve_qp(H, f, A, b)

    return u


def measure_config(_robot):
    return _robot.q


def send_joint_acceleration(_robot, _u, _qdot, _t, dt=0.01):
    # u is now 'qddot' (q double dot)
    _qdot += _u*dt
    q = _robot.q + _qdot*dt
    _robot.add_ani_frame(time=_t+dt, q = q)


def send_joint_velocity(_robot, _u, _t, dt=0.01):
    q_dot = _u
    q = _robot.q + q_dot*dt
    _robot.add_ani_frame(time=_t+dt, q = q)


In [12]:
import numpy as np
import uaibot as ub
import matplotlib.pyplot as plt

#Create the curve as a list of (n,1)-shaped np.matrix
#In this case, n=2
height = 0.5
center = [0.75, 0, height]
radius = 0.15
n_points = 1000


#Set the starting point
q = measure_config(robot)
jac, htm_e = robot.jac_geo(q)

#Set the integration step
dt = 0.01
#Set the simulation time
t_sim = 10

t = 0
htm_tg = ub.Utils.trn(np.random.rand(3,1)+0.1)
frame2 = ub.Frame(htm = htm_tg, size = 0.1) # fisrt target in first position
sim2 = ub.Simulation(
    [robot, frame2],
    width=600,
    height=600
)

n = robot.q.shape[0] # n dof
qdot = np.zeros((n,1))
q0 = np.random.rand(n,1)

total_sim_points = round(t_sim/dt)
for i in range(total_sim_points):

    q = measure_config(robot)
    jac, htm_e = robot.jac_geo(q)
    eff_pos = htm_e[0:3, 3]

    u = compute_control(q, qdot, robot, htm_tg)
    send_joint_acceleration(robot, u, qdot, t)
    t += dt


/tmp/ipykernel_2933735/1020138514.py:28: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  r[3,0] = 1-xd.T*x
/tmp/ipykernel_2933735/1020138514.py:29: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  r[4,0] = 1-yd.T*y
/tmp/ipykernel_2933735/1020138514.py:30: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  r[5,0] = 1-zd.T*z


In [13]:
if save:
    sim2.save(".", "problem2")
sim2.run()

# Problem 3

# Implement a code that samples 3 random spheres in random positions in the box given by 
, 
 and 
 and with a random radius. Ensure that no two spheres are colliding with each other. Sample also a random starting position and a random goal position within these boundaries.

Assume velocity control 
. Create a CBF controller to control the 3D position of a robot, modelled as a sphere with radius 
, to go the target position while avoiding collision between with the spheres and the boundaries of the box 
.

In [14]:
from typing import List

# This 'robot' is actually going to be a sphere (or Ball)
def send_joint_velocity(_robot, _u, _t, dt=0.01):
    new_pos = _robot.htm[0:3, 3] + _u*dt
    _robot.add_ani_frame(time=_t+dt, htm = ub.Utils.trn(new_pos))

class Sphere:
    def __init__(self, pos, radius):
        self.pos = np.matrix(pos).reshape(3,1)
        self.radius = radius
    
    def __sub__(self, other: "Sphere"):
        return np.linalg.norm(self.pos - other.pos)
    
    def __add__(self, other):
        return self.pos + other
    
    def __str__(self):
        return f"Sphere(pos={self.pos}, radius={self.radius})"

def get_random_sphere(pos_limits, radius_limits):
    xmin, xmax = pos_limits[0]
    ymin, ymax = pos_limits[1]
    zmin, zmax = pos_limits[2]
    pos = [
        np.random.rand()*(xmax - xmin) + xmin,
        np.random.rand()*(ymax - ymin) + ymin,
        np.random.rand()*(zmax - zmin) + zmin
    ]
    rmin, rmax = radius_limits
    return Sphere(np.matrix(pos), np.random.rand()*(rmax - rmin) + rmin)

def get_random_start_and_goal(box_limits, spheres, robot_radius=0.3):
    
    xmin, xmax = box_limits[0]
    ymin, ymax = box_limits[1]
    zmin, zmax = box_limits[2]
    start = Sphere(np.random.rand(3,1)*(xmax - xmin) + xmin, robot_radius)
    goal = Sphere(np.random.rand(3,1)*(xmax - xmin) + xmin, robot_radius)
    max_iter = 1000
    i = 0
    print(f"start = {start}")
    print(f"goal = {goal}")
    while np.linalg.norm(start- goal) < 0.5*(xmax - xmin) or is_colliding(start, spheres) or is_colliding(goal, spheres):
        goal = Sphere(np.random.rand(3,1)*(xmax - xmin) + xmin, robot_radius)
        i += 1
        if i > max_iter:
            raise ValueError("Failed to find a valid start and goal")
    return start, goal

def is_colliding(sphere: Sphere, spheres: List[Sphere]):
    for s in spheres:
        if np.linalg.norm(sphere.pos - s.pos) <= sphere.radius + s.radius:
            return True
    return False


def task_fun(robot, q, q_goal):

    r = q - q_goal
    jac_r = np.eye(3)
    return r, jac_r

def compute_control(robot, robot_htm, q_goal, spheres, box_limits, delta=0.1, K=1, eta=1):
    # Delta is the safety margin
    # K is the gain
    # eta is part of the CBF
    
    q = robot_htm[0:3, 3]
    
    r, jac_r = task_fun(robot, q, q_goal)

    # Add linear constraints (box)
    A = np.array([
        [1, 0, 0],
        [-1, 0, 0],
        [0, 1, 0],
        [0, -1, 0],
        [0, 0, 1],
        [0, 0, -1]
    ]).reshape(6,3)

    # Pay attention to the signal !
    b = np.array([
        box_limits[0][0],
        -box_limits[0][1],
        box_limits[1][0],
        -box_limits[1][1],
        box_limits[2][0],
        -box_limits[2][1],
    ]).reshape(6,1)

    num_spheres = len(spheres)
    
    # B = ||q - sphere_i - R1 - R2 - delta||²
    # grad_B = (q - sphere_i).T/|q - sphere_i|
    # B = np.zeros((num_spheres, 3))
    # grad_B = np.zeros((num_spheres, 3))

    for i, s in enumerate(spheres):
        # print(f"robot.radius = {robot.radius}")
        # print(f"s.radius = {s.radius}")
        # print(f"delta = {delta}")
        # print(f"np.linalg.norm(q - s.pos) = {np.linalg.norm(q - s.pos)}")
        # print(f"np.linalg.norm(q - s.pos).shape = {np.linalg.norm(q - s.pos).shape}")
        B_k_fun = np.linalg.norm(q - s.pos) - robot.radius - s.radius - delta
        # print(f"B_k_fun = {B_k_fun}")
        # print(f"B_k_fun.shape = {B_k_fun.shape}")
        # print(f"q = {q}")
        # print(f"s.pos = {s.pos}")
        # print(f"q - s.pos = {q - s.pos}")
        # print(f"np.linalg.norm(q - s.pos) = {np.linalg.norm(q - s.pos)}")
        grad_B_k_fun = (q - s.pos).T/np.linalg.norm(q - s.pos)
    
        A = np.vstack( (A, grad_B_k_fun) )
        b = np.vstack( (b, -eta * B_k_fun) )

    # Calculate H and f
    # H = dr/dq.T * dr/dq
    # print(f"jac_r.shape = {jac_r.shape}")
    H = jac_r.T * jac_r
    f = -2*jac_r.T * (-K*r)

    # # Solve QP
    # print(f"H.shape = {H.shape}")
    # print(f"f.shape = {f.shape}")
    # print(f"A.shape = {A.shape}")
    # print(f"b.shape = {b.shape}")
    u = ub.Utils.solve_qp(H, f, A, b)

    return u

    
# Initialization
spheres = []
i = 0
max_iter = 1000
box_size = 2
box_limits = [
    (-box_size, box_size),
    (-box_size, box_size),
    (-box_size, box_size)
]

radius_limits = (0.1, 1.0)
num_spheres = 15
# Select starting spheres
while i <= max_iter:
    sphere = get_random_sphere(box_limits, radius_limits)
    print(f"Got sphere: {sphere}")
    
    if len(spheres) == 0:
        spheres.append(sphere)
        print(f"Adding first sphere: {sphere}")
    elif not is_colliding(sphere, spheres):
        print(f"Adding sphere: {sphere}")
        spheres.append(sphere)
        if len(spheres) == num_spheres:
            print(f"Got {num_spheres} spheres, breaking")
            break
    else:
        print(f"Skipping sphere: {sphere} since it's colliding with: {spheres}") 
    
    # keep looping
    i += 1

start, goal = get_random_start_and_goal(box_limits, spheres, robot_radius=0.3)

print("Starting spheres:")
for s in spheres:
    print(f"Sphere: {s}")

print(f"Start: {start}")
print(f"Goal: {goal}")

# Main control loop
robot = ub.simobjects.Ball(
    htm=ub.Utils.trn(start.pos),
    radius=start.radius,
    color="yellow"
)
robot.set_ani_frame(htm=ub.Utils.trn(start.pos))

start_sphere = ub.simobjects.Ball(
    htm=ub.Utils.trn(start.pos),
    radius=robot.radius/2,
    color="blue"
)
start_sphere.set_ani_frame(htm=ub.Utils.trn(start.pos))

goal_sphere = ub.simobjects.Ball(
    htm=ub.Utils.trn(goal.pos),
    radius=robot.radius/2,
    color="green"
)
goal_sphere.set_ani_frame(htm=ub.Utils.trn(goal.pos))

sim_balls = []

for s in spheres:
    ball = ub.simobjects.Ball(
        htm=ub.Utils.trn(s.pos),
        radius=s.radius,
        color="purple"
    )
    ball.set_ani_frame(htm=ub.Utils.trn(s.pos))
    sim_balls.append(ball)

sim3 = ub.Simulation(
    [robot, start_sphere, goal_sphere] + sim_balls,
    width = 1000,
    height = 1000
)

t = 0
i = 0
dt = 0.01
max_iter = 10000
# Seed allows us to have reproducible results
seed = 100
np.random.seed(seed)
while i <= max_iter and np.linalg.norm(robot.htm[0:3, 3] - goal.pos) > 0.01:
    u = compute_control(robot, robot.htm, goal.pos, spheres, box_limits)
    send_joint_velocity(robot, u, t)
    # print(f"i = {i}, Dist to target = {np.linalg.norm(robot.htm[0:3, 3] - goal.pos)}")
    t += dt
    i += 1



Got sphere: Sphere(pos=[[ 1.314 ]
 [ 0.7307]
 [-0.4834]], radius=0.24478914731195084)
Adding first sphere: Sphere(pos=[[ 1.314 ]
 [ 0.7307]
 [-0.4834]], radius=0.24478914731195084)
Got sphere: Sphere(pos=[[-1.5714]
 [ 0.4074]
 [-1.5178]], radius=0.947436101726238)
Adding sphere: Sphere(pos=[[-1.5714]
 [ 0.4074]
 [-1.5178]], radius=0.947436101726238)
Got sphere: Sphere(pos=[[ 0.8406]
 [-0.7405]
 [-0.9741]], radius=0.523459649566383)
Adding sphere: Sphere(pos=[[ 0.8406]
 [-0.7405]
 [-0.9741]], radius=0.523459649566383)
Got sphere: Sphere(pos=[[ 1.6477]
 [-1.301 ]
 [ 1.6192]], radius=0.9743644756046919)
Adding sphere: Sphere(pos=[[ 1.6477]
 [-1.301 ]
 [ 1.6192]], radius=0.9743644756046919)
Got sphere: Sphere(pos=[[-1.4537]
 [ 1.1001]
 [ 1.0644]], radius=0.6547363799945237)
Adding sphere: Sphere(pos=[[-1.4537]
 [ 1.1001]
 [ 1.0644]], radius=0.6547363799945237)
Got sphere: Sphere(pos=[[ 0.4209]
 [-0.7358]
 [-1.9231]], radius=0.7851680302393305)
Skipping sphere: Sphere(pos=[[ 0.4209]
 [-0.73

ValueError: Failed to find a valid start and goal

In [ ]:
if save:
    sim3.save(".", "problem3")
sim3.run()

# Problem 4
# This problem is a harder version of the previous problem.

Create a code that, instead of spheres, create 3 random polyhedras with 6 faces each. The polyhedras should be placed in random poses inside the same box 
 as before. Again, ensure that no two objects are colliding with each other.

Our robot now is a box with sides 
. Furthermore, instead of just the linear velocity, we can control its angular velocity vector 
 as well (e.g., it is a box-shaped omnidirectional drone).

Create a CBF controller to manipulate the two velocities 
 to go to the target point while avoiding collision with the objects.

Remember that you can compute the distances and witness points, necessary for implementing the CBFs, by using the UAIBot's function compute_dist. Remember also that this function computes the distance, instead of the half-squared distance.

In [ ]:
from typing import List

# Update robot position and orientation based on linear and angular velocities
def send_joint_velocity(_robot, _u, _t, dt=0.01):
    """
    _u is a 6x1 vector: [v_x, v_y, v_z, ω_x, ω_y, ω_z]^T
    v is linear velocity, ω is angular velocity
    """
    v = _u[0:3]  # linear velocity
    w = _u[3:6]  # angular velocity
    
    # Update position
    new_pos = _robot.htm[0:3, 3] + v * dt
    
    # Update orientation using exponential map for small rotations
    # For small dt, R_new ≈ R_old * (I + [ω]_× * dt)
    w_skew = np.matrix([
        [0, -w[2,0], w[1,0]],
        [w[2,0], 0, -w[0,0]],
        [-w[1,0], w[0,0], 0]
    ])
    R_old = _robot.htm[0:3, 0:3]
    R_new = R_old * (np.eye(3) + w_skew * dt)
    
    # Normalize rotation matrix (Gram-Schmidt)
    u1 = R_new[:, 0] / np.linalg.norm(R_new[:, 0])
    u2 = R_new[:, 1] - (u1.T * R_new[:, 1])[0,0] * u1
    u2 = u2 / np.linalg.norm(u2)
    u3 = np.cross(u1.T, u2.T).T
    R_new = np.hstack([u1, u2, u3])
    
    # Create new HTM
    new_htm = np.eye(4)
    new_htm[0:3, 0:3] = R_new
    new_htm[0:3, 3] = np.asarray(new_pos).reshape(3,)
    
    _robot.add_ani_frame(time=_t+dt, htm=new_htm)

class ObstacleBox:
    def __init__(self, htm, width, depth, height):
        self.htm = htm
        self.width = width
        self.depth = depth
        self.height = height
        self.box_obj = ub.simobjects.Box(
            htm=htm,
            width=width,
            depth=depth,
            height=height,
            color="purple"
        )
    
    def __str__(self):
        pos = self.htm[0:3, 3]
        return f"ObstacleBox(pos={pos.T}, w={self.width}, d={self.depth}, h={self.height})"

def get_random_box(box_limits, size_limits):
    """Generate a random box with random pose and size"""
    xmin, xmax = box_limits[0]
    ymin, ymax = box_limits[1]
    zmin, zmax = box_limits[2]
    
    # Random position
    pos = np.matrix([
        [np.random.rand()*(xmax - xmin) + xmin],
        [np.random.rand()*(ymax - ymin) + ymin],
        [np.random.rand()*(zmax - zmin) + zmin]
    ])
    
    # Random orientation (Euler angles)
    alpha = np.random.rand() * 2 * np.pi
    beta = np.random.rand() * 2 * np.pi
    gamma = np.random.rand() * 2 * np.pi
    R = ub.Utils.rotz(alpha) * ub.Utils.roty(beta) * ub.Utils.rotx(gamma)
    
    # Random size
    smin, smax = size_limits
    width = np.random.rand()*(smax - smin) + smin
    depth = np.random.rand()*(smax - smin) + smin
    height = np.random.rand()*(smax - smin) + smin
    
    htm = ub.Utils.trn(pos) * R
    
    return ObstacleBox(htm, width, depth, height)

def is_colliding_box(box1: ObstacleBox, box2: ObstacleBox, margin=0.1):
    """Check if two boxes are colliding using compute_dist"""
    a_star, b_star, dist, _ = box1.box_obj.compute_dist(box2.box_obj)
    return dist < margin

def get_random_start_and_goal(box_limits, obstacles, robot_size):
    """Generate random start and goal positions that don't collide with obstacles"""
    xmin, xmax = box_limits[0]
    ymin, ymax = box_limits[1]
    zmin, zmax = box_limits[2]
    
    # Create temporary robot boxes for collision checking
    max_iter = 1000
    i = 0
    
    while i < max_iter:
        start_pos = np.matrix([
            [np.random.rand()*(xmax - xmin) + xmin],
            [np.random.rand()*(ymax - ymin) + ymin],
            [np.random.rand()*(zmax - zmin) + zmin]
        ])
        goal_pos = np.matrix([
            [np.random.rand()*(xmax - xmin) + xmin],
            [np.random.rand()*(ymax - ymin) + ymin],
            [np.random.rand()*(zmax - zmin) + zmin]
        ])
        
        # Check if start and goal are far enough apart
        if np.linalg.norm(start_pos - goal_pos) < 0.5*(xmax - xmin):
            i += 1
            continue
        
        # Create temporary boxes for collision checking
        start_box = ub.simobjects.Box(
            htm=ub.Utils.trn(start_pos),
            width=robot_size[0],
            depth=robot_size[1],
            height=robot_size[2]
        )
        goal_box = ub.simobjects.Box(
            htm=ub.Utils.trn(goal_pos),
            width=robot_size[0],
            depth=robot_size[1],
            height=robot_size[2]
        )
        
        # Check collisions with obstacles
        start_collision = False
        goal_collision = False
        
        for obs in obstacles:
            _, _, dist_start, _ = start_box.compute_dist(obs.box_obj)
            _, _, dist_goal, _ = goal_box.compute_dist(obs.box_obj)
            if dist_start < 0.1:
                start_collision = True
            if dist_goal < 0.1:
                goal_collision = True
        
        if not start_collision and not goal_collision:
            return start_pos, goal_pos
        
        i += 1
    
    raise ValueError("Failed to find a valid start and goal")

def task_fun(robot_pos, q_goal):
    """Task function: minimize distance to goal"""
    r = robot_pos - q_goal
    # Jacobian for position control (3x6: 3 for position, 6 for [v, ω])
    jac_r = np.hstack([np.eye(3), np.zeros((3, 3))])
    return r, jac_r

def compute_control(robot, robot_htm, q_goal, obstacles, box_limits, delta=0.1, K=1.0, eta=1.0):
    """
    Compute CBF controller for box robot with linear and angular velocity control.
    
    Control input u is 6x1: [v_x, v_y, v_z, ω_x, ω_y, ω_z]^T
    """
    q = robot_htm[0:3, 3]  # robot position
    S_A = q  # reference point on robot (center/origin)
    
    # Task function: minimize distance to goal
    r, jac_r = task_fun(q, q_goal)
    
    # QP objective: min ||jac_r * u + K*r||^2 + eps*||u||^2
    # H = jac_r^T * jac_r + eps*I, f = 2 * jac_r^T * K*r
    eps_reg = 0.01  # small regularization to ensure positive definiteness
    H = jac_r.T @ jac_r + eps_reg * np.eye(6)
    f = 2 * jac_r.T * (K * r)
    
    # Initialize constraint matrices (start empty, will add CBF constraints)
    A = np.matrix(np.zeros((0, 6)))  # 0 constraints initially, 6 control inputs
    b = np.matrix(np.zeros((0, 1)))  # 0 constraint bounds initially
    
    # CBF constraints for each obstacle
    for obs in obstacles:
        # Compute witness points using compute_dist
        a_star, b_star, dist, _ = robot.compute_dist(obs.box_obj)
        
        # Barrier function: B = dist - delta
        B_k = dist - delta
        
        # d/dt L_AB = (a* - b*)^T V_A + [(a* - S_A) x (a* - b*)^T] ω_A
        # For CBF: d/dt B >= -eta * B
        # So: grad_B^T [v; ω] >= -eta * B
        
        c_star = a_star - b_star  # direction vector from obstacle to robot
        
        # Linear velocity contribution: (a* - b*)^T * v
        grad_v = c_star.T
        
        # Angular velocity contribution: [(a* - S_A) x (a* - b*)]^T * ω
        # [(a* - S_A) x (a* - b*)^T] ω_A
        vec_a_minus_sa = a_star - S_A
        # Convert to arrays for cross product (np.cross expects 1D arrays)
        vec1 = np.array(vec_a_minus_sa).flatten()
        vec2 = np.array(c_star).flatten()
        cross_product = np.cross(vec1, vec2)
        # Convert back to matrix and reshape to row vector (1x3)
        grad_w = np.matrix(cross_product).reshape(1, 3)
        
        # Combined gradient: [grad_v, grad_w] (1x6)
        grad_B = np.hstack([grad_v, grad_w])
        
        # Constraint: grad_B^T * u >= -eta * B
        # In QP form: -grad_B^T * u <= eta * B
        # So: A_constraint = -grad_B^T, b_constraint = eta * B
        A_constraint = -grad_B
        b_constraint = np.matrix([[-eta * B_k]])
        
        A = np.vstack([A, A_constraint])
        b = np.vstack([b, b_constraint])
    
    # Solve QP: min (1/2) u^T H u + f^T u subject to A u >= b
    u = ub.Utils.solve_qp(H, f, A, b)
    
    return u

# Initialization
box_size = 2
box_limits = [
    (-box_size, box_size),
    (-box_size, box_size),
    (-box_size, box_size)
]

size_limits = (0.2, 0.8)  # size range for obstacles
num_obstacles = 30
obstacles = []
i = 0
max_iter = 1000

# Create random obstacles (boxes)
seed = 103
np.random.seed(seed)
while i < max_iter and len(obstacles) < num_obstacles:
    new_box = get_random_box(box_limits, size_limits)
    
    if len(obstacles) == 0:
        obstacles.append(new_box)
        print(f"Adding first obstacle: {new_box}")
    else:
        # Check collision with existing obstacles
        collision = False
        for obs in obstacles:
            if is_colliding_box(new_box, obs, margin=0.2):
                collision = True
                break
        
        if not collision:
            obstacles.append(new_box)
            print(f"Adding obstacle: {new_box}")
    
    i += 1

if len(obstacles) < num_obstacles:
    print(f"Warning: Only created {len(obstacles)} obstacles")

# Robot dimensions: lx=0.2, ly=0.3, lz=0.4
robot_size = (0.2, 0.3, 0.4)

# Get random start and goal
start_pos, goal_pos = get_random_start_and_goal(box_limits, obstacles, robot_size)
print(f"Start position: {start_pos.T}")
print(f"Goal position: {goal_pos.T}")

# Create robot
robot = ub.simobjects.Box(
    htm=ub.Utils.trn(start_pos),
    width=robot_size[0],   # lx
    depth=robot_size[1],   # ly
    height=robot_size[2],  # lz
    color="yellow"
)
robot.set_ani_frame(htm=ub.Utils.trn(start_pos))

# Create visualization markers
start_marker = ub.simobjects.Ball(
    htm=ub.Utils.trn(start_pos),
    radius=0.1,
    color="blue"
)
start_marker.set_ani_frame(htm=ub.Utils.trn(start_pos))

goal_marker = ub.simobjects.Ball(
    htm=ub.Utils.trn(goal_pos),
    radius=0.1,
    color="green"
)
goal_marker.set_ani_frame(htm=ub.Utils.trn(goal_pos))

# Set up obstacles in simulation
for obs in obstacles:
    obs.box_obj.set_ani_frame(htm=obs.htm)

# Create simulation
sim_objects = [robot, start_marker, goal_marker] + [obs.box_obj for obs in obstacles]
sim4 = ub.Simulation(sim_objects, width=600, height=600)

# Control loop
t = 0
dt = 0.01
i = 0
max_iter = 10000
tol = 0.05

while i < max_iter and np.linalg.norm(robot.htm[0:3, 3] - goal_pos) > tol:
    u = compute_control(robot, robot.htm, goal_pos, obstacles, box_limits, delta=0.1, K=1.0, eta=1.0)
    send_joint_velocity(robot, u, t, dt)
    
    if i % 100 == 0:
        dist_to_goal = np.linalg.norm(robot.htm[0:3, 3] - goal_pos)
        print(f"Step {i}: Distance to goal = {dist_to_goal:.4f}")
    
    t += dt
    i += 1

print(f"Finished after {i} steps. Final distance: {np.linalg.norm(robot.htm[0:3, 3] - goal_pos):.4f}")



In [ ]:
if save:
    sim4.save(".", "problem4")
sim4.run()